# 🎬 MovieLens Exploratory Data Analysis

ALS collaborative filtering — PySpark + pandas + matplotlib

In [ ]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, LongType, StructField, StructType

warnings.filterwarnings('ignore')
plt.style.use('dark_background')
sns.set_palette('husl')

spark = (
    SparkSession.builder
    .appName('MovieLens-EDA')
    .master('local[*]')
    .config('spark.driver.memory', '4g')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('SparkSession ready:', spark.version)

# Resolve data paths — works from notebooks/ or project root
BASE = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()

def find_csv(name):
    candidates = [
        os.path.join(BASE, 'data', name),
        os.path.join(BASE, 'data', 'ml-latest-small', name),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'{name} not found. Run: cd data && bash download_movielens.sh')

RATINGS_PATH = find_csv('ratings.csv')
MOVIES_PATH  = find_csv('movies.csv')
print('Ratings:', RATINGS_PATH)
print('Movies :', MOVIES_PATH)

## 1. Dataset Overview

In [ ]:
ratings_schema = StructType([
    StructField('userId',    IntegerType(), False),
    StructField('movieId',   IntegerType(), False),
    StructField('rating',    FloatType(),   False),
    StructField('timestamp', LongType(),    True),
])

ratings = spark.read.option('header', 'true').schema(ratings_schema).csv(RATINGS_PATH)
movies  = spark.read.option('header', 'true').csv(MOVIES_PATH)

print(f'Ratings shape  : {ratings.count():,} rows × {len(ratings.columns)} cols')
print(f'Movies shape   : {movies.count():,} rows × {len(movies.columns)} cols')
print(f'Unique users   : {ratings.select("userId").distinct().count():,}')
print(f'Unique movies  : {ratings.select("movieId").distinct().count():,}')
ratings.show(5)
movies.show(5)

## 2. Rating Distribution

In [ ]:
ratings_pd = ratings.select('rating').toPandas()

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(ratings_pd['rating'], bins=[0.25, 0.75, 1.25, 1.75, 2.25, 2.75, 3.25, 3.75, 4.25, 4.75, 5.25],
        color='#e94560', edgecolor='black', linewidth=0.5)
ax.set_xlabel('Rating', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Rating Distribution', fontsize=14, fontweight='bold')
ax.xaxis.set_ticks([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])

mean_r   = ratings_pd['rating'].mean()
median_r = ratings_pd['rating'].median()
ax.axvline(mean_r,   color='#f8b739', linestyle='--', label=f'Mean {mean_r:.2f}')
ax.axvline(median_r, color='#05c46b', linestyle='--', label=f'Median {median_r:.2f}')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Mean rating: {mean_r:.3f}   Median: {median_r:.3f}')

## 3. Top 10 Most Rated Movies

In [ ]:
top_rated = (
    ratings
    .groupBy('movieId')
    .agg(F.count('rating').alias('num_ratings'))
    .join(movies.select(F.col('movieId').cast(IntegerType()), 'title'), on='movieId')
    .orderBy(F.desc('num_ratings'))
    .limit(10)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_rated['title'][::-1], top_rated['num_ratings'][::-1], color='#0f3460')
for bar in bars:
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{int(bar.get_width()):,}', va='center', fontsize=9)
ax.set_xlabel('Number of Ratings')
ax.set_title('Top 10 Most Rated Movies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Top 10 Highest Average-Rated Movies (≥ 50 ratings)

In [ ]:
top_avg = (
    ratings
    .groupBy('movieId')
    .agg(
        F.count('rating').alias('num_ratings'),
        F.round(F.avg('rating'), 3).alias('avg_rating')
    )
    .filter(F.col('num_ratings') >= 50)
    .join(movies.select(F.col('movieId').cast(IntegerType()), 'title'), on='movieId')
    .orderBy(F.desc('avg_rating'))
    .limit(10)
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_avg['title'][::-1], top_avg['avg_rating'][::-1], color='#e94560')
for bar in bars:
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.2f}', va='center', fontsize=9)
ax.set_xlabel('Average Rating')
ax.set_xlim(3.5, 5.0)
ax.set_title('Top 10 Highest Avg-Rated Movies (≥50 ratings)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Genre Popularity

In [ ]:
genre_counts = (
    movies
    .select(F.explode(F.split('genres', r'\|')).alias('genre'))
    .filter(F.col('genre') != '(no genres listed)')
    .groupBy('genre')
    .count()
    .orderBy(F.desc('count'))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(11, 5))
colours = plt.cm.tab20.colors
ax.bar(genre_counts['genre'], genre_counts['count'],
       color=colours[:len(genre_counts)])
ax.set_xlabel('Genre')
ax.set_ylabel('Number of Movies')
ax.set_title('Movies per Genre', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. User Activity Distribution

In [ ]:
user_counts = (
    ratings
    .groupBy('userId')
    .agg(F.count('rating').alias('num_ratings'))
    .toPandas()
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(user_counts['num_ratings'], bins=50, color='#533483', edgecolor='black', linewidth=0.3)
ax.set_xlabel('Ratings per User')
ax.set_ylabel('Number of Users')
ax.set_title('User Activity Distribution', fontsize=14, fontweight='bold')
p95 = np.percentile(user_counts['num_ratings'], 95)
ax.axvline(p95, color='#f8b739', linestyle='--', label=f'95th percentile ({int(p95)} ratings)')
ax.legend()
plt.tight_layout()
plt.show()

power_users = user_counts[user_counts['num_ratings'] >= p95]
print(f'Power users (top 5%): {len(power_users):,}  (≥ {int(p95)} ratings each)')

## 7. Ratings Over Time

In [ ]:
yearly = (
    ratings
    .withColumn('year', F.year(F.from_unixtime(F.col('timestamp').cast('long'))))
    .groupBy('year')
    .agg(F.count('rating').alias('num_ratings'))
    .orderBy('year')
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(yearly['year'], yearly['num_ratings'], marker='o', color='#e94560', linewidth=2)
ax.fill_between(yearly['year'], yearly['num_ratings'], alpha=0.2, color='#e94560')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Ratings')
ax.set_title('Ratings per Year', fontsize=14, fontweight='bold')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.show()

## 8. Matrix Sparsity

In [ ]:
n_users  = ratings.select('userId').distinct().count()
n_movies = ratings.select('movieId').distinct().count()
n_ratings = ratings.count()
total_cells = n_users * n_movies
sparsity = 1.0 - (n_ratings / total_cells)

print(f'Users          : {n_users:>10,}')
print(f'Movies         : {n_movies:>10,}')
print(f'Ratings        : {n_ratings:>10,}')
print(f'Total cells    : {total_cells:>10,}')
print(f'Sparsity       : {sparsity*100:>9.3f}%')
print(f'Density        : {(1-sparsity)*100:>9.3f}%')

## 9. ALS Quick Test

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType

sample = ratings.sample(fraction=0.3, seed=42).cache()
train, test = sample.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol='userId', itemCol='movieId', ratingCol='rating',
    rank=10, maxIter=5, regParam=0.1,
    coldStartStrategy='drop', nonnegative=True, seed=42
)
model = als.fit(train)

evaluator = RegressionEvaluator(metricName='rmse', labelCol='rating', predictionCol='prediction')
rmse = evaluator.evaluate(model.transform(test))
print(f'Quick-test RMSE: {rmse:.4f}')

# Top 5 recommendations for user 1
user_schema = StructType([F._create_column_from_literal(1).schema if False else
    __import__('pyspark.sql.types', fromlist=['StructField', 'StructType', 'IntegerType']).StructField('userId', __import__('pyspark.sql.types', fromlist=['IntegerType']).IntegerType(), False)])
user_df = spark.createDataFrame([(1,)], ['userId'])
recs = model.recommendForUserSubset(user_df, 5)
recs.select('userId', F.explode('recommendations').alias('rec')) \
    .select('userId', F.col('rec.movieId').alias('movieId'), F.round('rec.rating', 3).alias('score')) \
    .join(movies.select(F.col('movieId').cast(IntegerType()), 'title'), on='movieId') \
    .select('movieId', 'title', 'score') \
    .show(5, truncate=False)